# Tratamento gold — Chuva (Parquet)

Pega a série diária de chuva já limpa pela silver (`chuva_diaria.csv`) e
grava em **Parquet**, com cada coluna no tipo certo.

Entrada (silver): `data/silver/ana/chuva/chuva_diaria.csv`
Saída (gold):     `data/gold/ana/chuva/chuva_diaria.parquet`

**Por que Parquet em vez de CSV aqui:**
- Guarda o tipo de cada coluna junto com o arquivo (não precisa "adivinhar"
  toda vez que lê, como acontece com CSV).
- Arquivo bem menor e leitura bem mais rápida que CSV pra volume grande.
- Praticamente qualquer ferramenta de análise (pandas, Power BI, DuckDB,
  Excel via Power Query...) já sabe abrir direto, sem precisar de um banco
  de dados rodando.

Esse primeiro tratamento gold **não agrega nada ainda** (não soma por mês,
não calcula média por estação) — só padroniza tipo e formato. Agregações
específicas ficam melhor como um próximo notebook/script, construído em
cima desse Parquet.

Rode as células em ordem, de cima pra baixo.

In [ ]:
from pathlib import Path

import pandas as pd


def encontrar_raiz_do_projeto(marcador: str = "pyproject.toml") -> Path:
    """Sobe a partir da pasta atual até achar o pyproject.toml do projeto.

    Assim funciona não importa em qual pasta este notebook esteja de fato
    (src/gold/, data/silver/, ou qualquer outra) — evita o bug clássico
    de calcular a raiz errado quando o notebook é movido de lugar.
    """
    atual = Path.cwd().resolve()
    for pasta in [atual, *atual.parents]:
        if (pasta / marcador).exists():
            return pasta
    raise FileNotFoundError(
        f"Não encontrei '{marcador}' subindo a partir de {atual}. "
        "Este notebook está dentro da pasta do projeto?"
    )


RAIZ = encontrar_raiz_do_projeto()

ENTRADA_CSV = RAIZ / "data" / "silver" / "ana" / "chuva" / "chuva_diaria.csv"
SAIDA_PARQUET = RAIZ / "data" / "gold" / "ana" / "chuva" / "chuva_diaria.parquet"

print("Raiz do projeto:", RAIZ)
print("Lendo de:", ENTRADA_CSV)
print("Vai salvar em:", SAIDA_PARQUET)

## 1. Ler o CSV da silver

In [ ]:
silver = pd.read_csv(ENTRADA_CSV)
print(f"{len(silver)} linha(s) lida(s) da silver.")
silver.dtypes

## 2. Aplicar o schema final (tipo de cada coluna)

Uso os tipos "nullable" do pandas (`Int64`, com I maiúsculo) em vez dos
tipos nativos (`int64`) porque eles aceitam valor nulo — os tipos nativos
do NumPy não aceitam `NaN` em colunas inteiras.

In [ ]:
TIPOS_FINAIS = {
    "codigoestacao": "Int64",
    "chuva_mm": "float32",
    "status": "Int64",
    "nivel_consistencia": "Int64",
    "tipo_medicao": "Int64",
}

gold = silver.copy()
gold["data"] = pd.to_datetime(gold["data"]).dt.date

for coluna, tipo in TIPOS_FINAIS.items():
    gold[coluna] = pd.to_numeric(gold[coluna], errors="coerce").astype(tipo)

gold.dtypes

## 3. Conferência rápida

In [ ]:
print("Linhas:", len(gold))
print("Estações distintas:", gold["codigoestacao"].nunique())
print("Nulos por coluna:")
print(gold.isna().sum())
gold.head()

## 4. Salvar em Parquet (gold)

In [ ]:
SAIDA_PARQUET.parent.mkdir(parents=True, exist_ok=True)
gold.to_parquet(SAIDA_PARQUET, index=False, engine="pyarrow")

tamanho_csv_mb = ENTRADA_CSV.stat().st_size / 1e6
tamanho_parquet_mb = SAIDA_PARQUET.stat().st_size / 1e6
print(f"Salvo em {SAIDA_PARQUET}")
print(f"Tamanho: {tamanho_csv_mb:.1f} MB (csv) -> {tamanho_parquet_mb:.1f} MB (parquet)")

## Próximos passos

- Replicar essa mesma lógica pra cota, depois que `tratar_cota` (silver)
  existir.
- Um próximo notebook gold pode agregar (ex.: total mensal por estação) ou
  já juntar chuva + cota + El Niño num único Parquet de análise.